# 04 — Engineer Time Series
Aggregates scored articles into weekly time series per subtopic.
Input: `data/scored_articles.csv` → Output: `data/timeseries.csv`

In [1]:
import pandas as pd

df = pd.read_csv('data/scored_articles.csv', parse_dates=['date'])
df = df.sort_values('date')
print(df.shape)

(12447, 11)


In [2]:
# Resample per week per subtopic
df['week'] = df['date'].dt.to_period('W').dt.start_time

ts = (
    df.groupby(['week', 'subtopic'])
    .agg(
        article_count=('id', 'count'),
        sentiment_mean=('sentiment', 'mean'),
        sentiment_std=('sentiment', 'std'),
        positive_ratio=('sentiment_label', lambda x: (x == 'positive').mean()),
        negative_ratio=('sentiment_label', lambda x: (x == 'negative').mean()),
    )
    .reset_index()
)

ts['sentiment_std'] = ts['sentiment_std'].fillna(0)
print(ts.shape)
ts.head(10)

(2801, 7)


,week,subtopic,article_count,sentiment_mean,sentiment_std,positive_ratio,negative_ratio
0,2020-12-28,Energy & Environment,1,-0.5423,0.000000,0.000000,1.000000
1,2020-12-28,Flash news,1,0.0000,0.000000,0.000000,0.000000
2,2020-12-28,Healthcare,1,0.0000,0.000000,0.000000,0.000000
3,2020-12-28,Politics,3,-0.3454,0.300102,0.000000,0.666667
4,2020-12-28,Supply Chain,1,-0.5423,0.000000,0.000000,1.000000
5,2020-12-28,Weather & Animal,1,-0.8720,0.000000,0.000000,1.000000
6,2021-01-04,AI Safety,1,-0.6249,0.000000,0.000000,1.000000
7,2021-01-04,Big Tech,1,0.4215,0.000000,1.000000,0.000000
8,2021-01-04,Energy & Environment,1,0.8957,0.000000,1.000000,0.000000
9,2021-01-04,Flash news,3,-0.1881,0.533908,0.333333,0.333333


In [3]:
# Pivot to wide format for easy plotting (optional)
# Each subtopic becomes a column group
ts_wide = ts.pivot_table(index='week', columns='subtopic', values='article_count', fill_value=0)
print('Wide format shape:', ts_wide.shape)
ts_wide.head()

Wide format shape: (266, 12)


subtopic,AI Safety,Big Tech,Energy & Environment,Flash news,Healthcare,Legal,OpenAI,Politics,Social Media,Stocks market,Supply Chain,Weather & Animal
week,,,,,,,,,,,,
2020-12-28,0.0,0.0,1.0,1.0,1.0,0.0,0.0,3.0,0.0,0.0,1.0,1.0
2021-01-04,1.0,1.0,1.0,3.0,0.0,2.0,0.0,3.0,1.0,4.0,6.0,0.0
2021-01-11,2.0,1.0,1.0,4.0,2.0,0.0,1.0,3.0,2.0,1.0,1.0,1.0
2021-01-18,0.0,2.0,0.0,2.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2021-01-25,1.0,5.0,0.0,3.0,1.0,0.0,2.0,0.0,6.0,1.0,0.0,2.0


In [4]:
ts.to_csv('data/timeseries.csv', index=False)
ts_wide.to_csv('data/timeseries_wide.csv')
print('Saved: data/timeseries.csv and data/timeseries_wide.csv')

Saved: data/timeseries.csv and data/timeseries_wide.csv
